# 04. Regression: what predicts county Medicare spending?

**Model** (one row per county-year, 2014-2024):

spend_pc_std = b1·income + b2·poverty + b3·rural + b4·avg_age + b5·dual% + b6·MA% + year fixed effects + error

* Standard errors clustered by state (50 clusters).
* **Robustness check:** the same model weighted by number of beneficiaries (WLS),
  so a county with 100,000 beneficiaries counts more than one with 600.

Stata equivalents, for reference:

```stata
reg spend_pc_std income_10k poverty_rate rural avg_age dual_pct ma_rate i.year, vce(cluster state)
reg spend_pc_std income_10k poverty_rate rural avg_age dual_pct ma_rate i.year [aw=benes], vce(cluster state)
```

These are associations across counties, not causal effects.

Later sections look at the spending the model can't explain (section 4) and whether
higher spending goes with better outcomes (section 5).

In [ ]:
import duckdb
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

## 1. Data

Variables are rescaled in SQL so coefficients are easy to read. `m.` and `r.` say
which table each column comes from (`merged` and `raw`), since both have a year column:

| Variable | Unit |
|---|---|
| `income_10k` | median household income in $10,000s |
| `poverty_rate` | percentage points (already) |
| `rural` | 1 = nonmetro (RUCC 4-9), 0 = metro |
| `avg_age` | years |
| `dual_pct` | % of beneficiaries also on Medicaid, 0-100 |
| `ma_rate` | % of all Medicare enrollees in Medicare Advantage, 0-100 |

In [ ]:
with duckdb.connect("../medicare.duckdb", read_only=True) as con:
    df = con.sql("""
        SELECT m.year, m.fips, m.county_name, m.state, m.benes, m.spend_pc_std,
               m.median_income / 10000 AS income_10k,
               m.poverty_rate,
               CASE m.metro_rural WHEN 'Rural' THEN 1 WHEN 'Metro' THEN 0 END AS rural,
               m.avg_age,
               m.dual_pct * 100 AS dual_pct,
               m.ma_rate  * 100 AS ma_rate,
               m.readmit_pct * 100 AS readmit,   -- 30-day readmission rate, %
               m.er_per_1000       AS er,        -- ER visits per 1,000 beneficiaries
               TRY_CAST(r.IP_MDCR_STDZD_PYMT_PC AS DOUBLE) AS ip_spend   -- inpatient hospital spending
        FROM merged m
        LEFT JOIN raw r
          ON r.BENE_GEO_LVL = 'County' AND r.BENE_AGE_LVL = 'All'
         AND r.YEAR = m.year AND r.BENE_GEO_CD = m.fips
    """).df()

# statsmodels drops rows with missing values inside the regression, but the
# cluster groups must line up row for row, so drop them up front instead.
model_vars = ["spend_pc_std", "income_10k", "poverty_rate", "rural", "avg_age", "dual_pct", "ma_rate"]
reg = df.dropna(subset=model_vars).copy()
print(f"{len(df):,} rows, {len(reg):,} with complete data, {reg['state'].nunique()} states")

In [ ]:
reg[model_vars].describe().T.round(2)

## 2. Estimate

`C(year)` adds a dummy for each year (like `i.year` in Stata). It absorbs
anything that affects every county in a given year: national payment updates,
inflation, the 2020 COVID drop.

In [ ]:
formula = "spend_pc_std ~ income_10k + poverty_rate + rural + avg_age + dual_pct + ma_rate + C(year)"
clusters = {"groups": reg["state"]}

ols = smf.ols(formula, data=reg).fit(cov_type="cluster", cov_kwds=clusters)
wls = smf.wls(formula, data=reg, weights=reg["benes"]).fit(cov_type="cluster", cov_kwds=clusters)

main_vars = ["income_10k", "poverty_rate", "rural", "avg_age", "dual_pct", "ma_rate"]
print(summary_col(
    [ols, wls], model_names=["OLS", "WLS (weighted by benes)"], stars=True,
    regressor_order=main_vars, drop_omitted=True,
    info_dict={"N": lambda r: f"{int(r.nobs):,}", "Year FE": lambda r: "Yes"},
))

Side by side with 95% confidence intervals:

In [ ]:
def tidy(res, name):
    ci = res.conf_int().loc[main_vars]
    return pd.DataFrame({"model": name, "coef": res.params[main_vars], "se": res.bse[main_vars],
                         "ci_low": ci[0], "ci_high": ci[1], "p": res.pvalues[main_vars]})

coefs = pd.concat([tidy(ols, "OLS"), tidy(wls, "WLS")])
coefs.round(3)

## 3. Extra robustness: drop Miller County, GA

Miller County is the highest-spending county in 2023-2024 (about $30K per
beneficiary, driven by skilled nursing facility spending). Does one county move the results?

In [ ]:
no_miller = reg[reg["fips"] != "13201"]
ols_nm = smf.ols(formula, data=no_miller).fit(cov_type="cluster", cov_kwds={"groups": no_miller["state"]})
pd.DataFrame({"OLS": ols.params[main_vars], "OLS without Miller": ols_nm.params[main_vars]}).round(1)

## 4. Spending the model can't explain

The regression predicts each county's spending from its population and rurality.
The **residual** (actual minus predicted) is what is left over: positive means a
county spends more than counties with similar characteristics, negative means less.
With an R-squared of about 0.42, more than half of the variation is unexplained, which
is the classic Dartmouth Atlas question: why do similar places spend so differently?

Residuals come from the unweighted OLS model, so they already account for the year.

In [ ]:
reg["predicted"] = ols.fittedvalues
reg["unexplained"] = ols.resid
r24 = reg[reg["year"] == 2024]

print(f"2024: {len(r24):,} counties, residual SD ${r24['unexplained'].std():,.0f}, "
      f"{(r24['unexplained'].abs() < 1000).mean():.0%} within $1,000 of prediction")
r24.nlargest(8, "unexplained")[["county_name", "benes", "spend_pc_std", "predicted", "unexplained"]].round(0)

In [ ]:
r24.nsmallest(5, "unexplained")[["county_name", "benes", "spend_pc_std", "predicted", "unexplained"]].round(0)

**Is it persistent?** If unexplained spending were mostly noise, a county's residual
in one period would say little about the next. Compare each county's average residual
in 2014-2018 with 2020-2024:

In [ ]:
early = reg[reg["year"] <= 2018].groupby("fips")["unexplained"].mean()
late = reg[reg["year"] >= 2020].groupby("fips")["unexplained"].mean()
both = pd.concat([early, late], axis=1, keys=["2014-2018", "2020-2024"]).dropna()
print(f"Correlation across {len(both):,} counties: {both.corr().iloc[0, 1]:.2f}")

By state (beneficiary-weighted average residual, 2024):

In [ ]:
by_state = (r24.assign(w=r24["unexplained"] * r24["benes"])
              .groupby("state")[["w", "benes"]].sum()
              .pipe(lambda d: (d["w"] / d["benes"]).rename("unexplained"))
              .sort_values())
pd.concat([by_state.head(6), by_state.tail(6)]).round(0)

In [ ]:
import json

import plotly.express as px

import plot_style
from plot_style import DIVERGING, subtitle, site_copy, write_site_chart

# Darker than usual: the diverging scale's midpoint is already light gray, so counties
# with no data need a clearly different shade.
NO_DATA_DARK = "#bdbbb4"

with open("../data/counties_2023.geojson") as f:
    counties_geo = json.load(f)

# Symmetric color range so zero sits exactly at the gray midpoint; capped at the
# 95th percentile of the absolute residual so a few extremes don't wash out the rest.
cap = round(r24["unexplained"].abs().quantile(0.95), -2)

fig_resid = px.choropleth(
    r24, geojson=counties_geo, locations="fips", color="unexplained",
    color_continuous_scale=DIVERGING, range_color=(-cap, cap), scope="usa",
    hover_name="county_name",
    hover_data={"fips": False, "unexplained": ":+$,.0f", "spend_pc_std": ":$,.0f", "predicted": ":$,.0f"},
    labels={"unexplained": "Unexplained", "spend_pc_std": "Actual", "predicted": "Predicted"},
    title=subtitle("Spending the model can't explain, 2024",
                   "Actual minus predicted spending per beneficiary. Red: more than similar counties. "
                   "Blue: less. Dark gray: not in the sample."),
)
fig_resid.update_traces(marker_line_width=0.2, marker_line_color="white")
fig_resid.update_geos(showland=True, landcolor=NO_DATA_DARK, showlakes=True, lakecolor=plot_style.SURFACE,
                      showsubunits=True, subunitcolor="white", subunitwidth=1, bgcolor=plot_style.SURFACE)
fig_resid.update_layout(
    height=600, margin=dict(l=10, r=10, t=90, b=10),
    coloraxis_colorbar=dict(title="", tickformat="+$,.0f", thickness=12, len=0.6),  # "+$3,000"
)
# Shown as a static image here: an interactive map embeds ~6 MB of county shapes in the
# notebook file. The interactive version is the website copy saved in the next cell.
fig_resid.show(renderer="png", width=1000, height=600)

In [ ]:
write_site_chart(site_copy(fig_resid), "unexplained")

## 5. Does more spending buy better outcomes?

If high-spending counties were buying better care, they should have fewer
avoidable bad events. The CMS file has two county-level measures (its
preventable-hospitalization rates are only published for states):

* **30-day readmission rate** (`readmit`, %): hospital patients readmitted within 30 days. Lower is better.
* **ER visits per 1,000 beneficiaries** (`er`)

First, a simple comparison: split 2024 counties into fifths by unexplained
spending (from section 4) and compare outcomes, weighting by beneficiaries.

In [ ]:
q = r24.dropna(subset=["readmit", "er"]).copy()
q["fifth"] = pd.qcut(q["unexplained"], 5, labels=["Lowest", "2nd", "Middle", "4th", "Highest"])


def wmean(d, col):
    return (d[col] * d["benes"]).sum() / d["benes"].sum()


by_fifth = q.groupby("fifth", observed=True).apply(lambda d: pd.Series({
    "counties": len(d),
    "unexplained": wmean(d, "unexplained"),
    "spending": wmean(d, "spend_pc_std"),
    "readmit": wmean(d, "readmit"),
    "er": wmean(d, "er"),
}))
by_fifth.round(1)

Now the same question as a regression, using all years: outcome on spending (in
$1,000s) plus the same controls and year effects, clustered by state. Three checks:

* **Weighted** by beneficiaries, as before.
* **Spending outside the hospital only.** A readmission is itself a hospital stay that
  costs money, so part of any link could be mechanical. Using spending minus inpatient
  hospital spending removes that channel.
* **County fixed effects** (`C(fips)`, like `areg ..., absorb(fips)` in Stata): compares
  each county with itself over time, removing every stable difference between counties.
  (Rurality never changes within a county, so it drops out; statsmodels warns about this.)

In [ ]:
import warnings

reg["spend_1k"] = reg["spend_pc_std"] / 1000
reg["nonip_1k"] = (reg["spend_pc_std"] - reg["ip_spend"]) / 1000
controls = "income_10k + poverty_rate + rural + avg_age + dual_pct + ma_rate + C(year)"

rows = []
for outcome in ["readmit", "er"]:
    for x, extra, label, weighted in [
        ("spend_1k", "", "OLS", False),
        ("spend_1k", "", "WLS (weighted)", True),
        ("nonip_1k", "", "OLS, non-hospital spending", False),
        ("spend_1k", " + C(fips)", "County fixed effects", False),
    ]:
        d = reg.dropna(subset=[outcome, x])
        f = f"{outcome} ~ {x} + {controls}{extra}"
        model = smf.wls(f, d, weights=d["benes"]) if weighted else smf.ols(f, d)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")  # rank warning from rural in the FE model
            res = model.fit(cov_type="cluster", cov_kwds={"groups": d["state"]})
        rows.append({"outcome": outcome, "model": label, "per_1k": res.params[x],
                     "se": res.bse[x], "p": res.pvalues[x], "N": int(res.nobs)})

outcomes = pd.DataFrame(rows)
outcomes.round(3)

In [ ]:
from plotly.subplots import make_subplots

from plot_style import INK_2, METRO

# Short labels so the two panels fit side by side on a phone.
fig_out = make_subplots(rows=1, cols=2, horizontal_spacing=0.14,
                        subplot_titles=("Readmission rate (%)", "ER visits per 1,000"))
short = ["Low", "2", "3", "4", "High"]
for col, (var, fmt) in enumerate([("readmit", "{:.1f}%"), ("er", "{:.0f}")], start=1):
    fig_out.add_bar(
        x=short, y=by_fifth[var], marker_color=METRO, showlegend=False,
        text=[fmt.format(v) for v in by_fifth[var]], textposition="outside", cliponaxis=False,
        textfont=dict(color=INK_2, size=12),
        customdata=by_fifth[["unexplained", "counties"]].to_numpy(),
        hovertemplate="Fifth: %{x}<br>Unexplained spending: %{customdata[0]:+$,.0f}<br>"
                      "%{customdata[1]} counties<extra></extra>",
        row=1, col=col,
    )
    fig_out.update_yaxes(range=[0, by_fifth[var].max() * 1.18], row=1, col=col)
fig_out.update_annotations(font=dict(size=13, color=INK_2))  # the two panel titles
fig_out.add_annotation(text="Counties in fifths by unexplained spending",
                       x=0.5, y=-0.13, xref="paper", yref="paper", showarrow=False,
                       font=dict(size=12, color=plot_style.MUTED))
fig_out.update_layout(
    height=440, bargap=0.4, margin=dict(t=110, b=70),
    title=subtitle("Counties that spend more than predicted have more readmissions and ER visits",
                   "2024, beneficiary-weighted averages. Fifths of counties by actual minus predicted spending."),
)
fig_out.show()

In [ ]:
web = site_copy(fig_out)
web.update_layout(margin_t=40)
write_site_chart(web, "outcomes")

## 6. Reading the results

*(Numbers below are from the run on 2026-11-02; rerun to confirm.)*

* **Rural:** after controls, rural counties spend about **$620 less** per
  beneficiary than metro counties (about $860 less when weighted). Significant at 1% in both.
* **Poverty:** each additional point of poverty is associated with about
  **$80-90 more** per beneficiary.
* **Income:** small and not significant once poverty is in the model. Income and
  poverty are correlated at about -0.74, so the model has a hard time separating them.
* **Age and dual eligibility:** both positive and significant. Dual-eligible
  beneficiaries tend to be sicker and lower income.
* **Medicare Advantage share:** close to zero and not significant.
* **Weighting** strengthens the rural gap, but signs and significance are the
  same, so the small counties are not driving the results.
* **Fit:** R² of about 0.42 (OLS) and 0.47 (WLS), including the year effects.
* **Unexplained spending is persistent:** counties that spent more than predicted in
  2014-2018 mostly still did in 2020-2024 (correlation about 0.88), so the leftover
  differences are not just year-to-year noise.
* **More spending does not buy better outcomes on these measures.** Each extra
  $1,000 per beneficiary goes with about 0.6 points more readmissions and about 30
  more ER visits per 1,000, holding population fixed. The link holds when weighting,
  when using only spending outside the hospital, and within counties over time. This is
  consistent with the Dartmouth Atlas finding that higher-spending regions do not get
  better care, though it does not show that spending causes worse outcomes.

**Caveat:** standardized spending removes price differences, not health
differences. Some of what looks like "rural counties spend less" could be
differences in health status or in access to care.